# 08 Strict Cutoff Sensitivity Check

The labels are for subscribers whose plan expires in March 2017, and churn means no new subscription within 30 days of that expiry.
The main pipeline uses every transaction up to 31 March, so for many subscribers the March renewal (or its absence) is already in the features.
That is allowed by the question as stated (predict on 31 March), but it makes the task easier than an early-warning task.

Here we move the billing cutoff to **1 March 2017**, before any label expiry, rebuild the billing features, and retrain the same tuned LightGBM
on the same train / validation / test split. Test data is only used for the final scores.

In [1]:
import json
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, f1_score, log_loss, precision_score, recall_score, roc_auc_score

RAW = Path("../data/raw")
PROCESSED = Path("../data/processed")
MODELS = Path("../models")
RESULTS = Path("../results")
STRICT = pd.Timestamp("2017-03-01")

cfg = json.loads((MODELS / "model_config.json").read_text())
GROUPS, CATEGORICAL, PARAMS = cfg["groups"], cfg["categorical"], cfg["best_params"]
df = pd.read_parquet(PROCESSED / "features_train.parquet").merge(pd.read_csv(PROCESSED / "split_assignments.csv"), on="msno")
print(df.shape, df.split.value_counts().to_dict())

(946686, 36) {'train': 662679, 'test': 189338, 'val': 94669}


## How much of the label is visible in March billing

In [2]:
tx = pd.read_csv(RAW / "transactions.csv")
tx = tx[tx.msno.isin(df.msno)]
tx["transaction_date"] = pd.to_datetime(tx.transaction_date.astype(str), format="%Y%m%d")
tx["membership_expire_date"] = pd.to_datetime(tx.membership_expire_date.astype(str), format="%Y%m%d")

march = df.msno.isin(tx.loc[tx.transaction_date >= STRICT, "msno"])
overlap = (df.assign(march_tx=march).groupby("march_tx").is_churn.agg(n="size", churn_rate="mean"))
overlap["share"] = overlap.n / len(df)
print(f"{(tx.transaction_date >= STRICT).mean():.1%} of transactions are dated in March 2017")
overlap.round(4)

85.0% of transactions are dated in March 2017


,n,churn_rate,share
march_tx,,,
False,43550,0.8088,0.046
True,903136,0.0553,0.954


## Billing Features from Transactions before 1 March

In [3]:
pre = tx[tx.transaction_date < STRICT].sort_values(["msno", "transaction_date", "membership_expire_date"])
pre["is_discount"] = (pre.actual_amount_paid < pre.plan_list_price).astype(int)
g = pre.groupby("msno")
bill = g.agg(tx_count=("is_cancel", "size"), total_paid=("actual_amount_paid", "sum"),
             avg_plan_days=("payment_plan_days", "mean"), auto_renew_ratio=("is_auto_renew", "mean"),
             cancel_count=("is_cancel", "sum"), discount_count=("is_discount", "sum"),
             n_payment_methods=("payment_method_id", "nunique"))
last = g.tail(1).set_index("msno")
bill = bill.assign(
    last_payment_method=last.payment_method_id, last_plan_days=last.payment_plan_days,
    last_amount_paid=last.actual_amount_paid, last_auto_renew=last.is_auto_renew, last_is_cancel=last.is_cancel,
    days_since_last_tx=(STRICT - last.transaction_date).dt.days,
    days_until_expiration=(last.membership_expire_date - STRICT).dt.days,
).reset_index()
del tx, pre, g, last

BILLING = GROUPS["billing"]
strict = df.drop(columns=BILLING).merge(bill, on="msno", how="left")
strict["has_tx"] = strict.tx_count.notna().astype(int)
zero = ["tx_count", "total_paid", "cancel_count", "discount_count", "n_payment_methods"]
strict[zero] = strict[zero].fillna(0)
strict[BILLING] = strict[BILLING].astype(np.float32)
print(f"subscribers with any billing record before 1 March: {strict.has_tx.mean():.1%}")

subscribers with any billing record before 1 March: 9.5%


## Retrain on the Same Split

Same tuned LightGBM hyperparameters as the champion; the number of trees is chosen by early stopping on validation. Three feature sets:
- **Strict billing + demographics**: nothing dated on or after 1 March.
- **Strict billing + demographics + March listening**: adds the listening features. Listening in March can still fall after a subscriber's expiry,
  but it contains no payment or renewal record.
- **Original (31 March) champion**, for comparison.

In [4]:
SETS = {"Strict billing + demographics": GROUPS["demographic"] + BILLING,
        "Strict billing + demographics + listening": GROUPS["demographic"] + BILLING + GROUPS["listening"]}
part = {s: strict[strict.split == s] for s in ["train", "val", "test"]}
y = {s: part[s].is_churn.values for s in part}

def fit(cols):
    m = lgb.LGBMClassifier(n_estimators=2000, random_state=42, verbose=-1, n_jobs=-1, **PARAMS)
    m.fit(part["train"][cols], y["train"], eval_X=(part["val"][cols],), eval_y=(y["val"],),
          categorical_feature=[c for c in CATEGORICAL if c in cols], callbacks=[lgb.early_stopping(50, verbose=False)])
    return m

probs, val_probs = {}, {}
for name, cols in SETS.items():
    m = fit(cols)
    val_probs[name] = m.predict_proba(part["val"][cols])[:, 1]
    probs[name] = m.predict_proba(part["test"][cols])[:, 1]
    print(f"{name:45s} trees {m.best_iteration_:4d} | val PR-AUC {average_precision_score(y['val'], val_probs[name]):.4f}")

# simple baselines under the same strict cutoff
rule_rates = pd.Series(y["train"]).groupby(part["train"].last_auto_renew.fillna(-1).values).mean()
probs["Rule: auto-renew off (strict)"] = part["test"].last_auto_renew.fillna(-1).map(rule_rates).fillna(y["train"].mean()).values
probs["Majority (prior)"] = np.full(len(y["test"]), y["train"].mean())

# original champion on the same test subscribers
X_test = pd.read_parquet(PROCESSED / "X_test.parquet")
champion = lgb.Booster(model_file=str(MODELS / "lgbm_champion.txt"))
orig = pd.Series(champion.predict(X_test[cfg["features"]]), index=X_test.msno.values)
probs["Original champion (31 March billing)"] = orig.loc[part["test"].msno].values

Strict billing + demographics                 trees  282 | val PR-AUC 0.4172


Strict billing + demographics + listening     trees  328 | val PR-AUC 0.5783


## Test Scores

The decision threshold for each strict model is chosen on validation (best F1), exactly as for the champion.

In [5]:
grid = np.round(np.arange(0.02, 0.96, 0.01), 2)
def tau_for(name):
    if name not in val_probs:
        return cfg["threshold"]
    return float(grid[int(np.argmax([f1_score(y["val"], val_probs[name] >= t) for t in grid]))])

rng = np.random.default_rng(0)
yt = y["test"]
boot_idx = [rng.integers(0, len(yt), len(yt)) for _ in range(500)]
rows = []
for name, p in probs.items():
    t = tau_for(name)
    pr_boot = [average_precision_score(yt[i], p[i]) for i in boot_idx]
    rows.append({"model": name, "pr_auc": average_precision_score(yt, p), "pr_auc_low": np.quantile(pr_boot, 0.025),
                 "pr_auc_high": np.quantile(pr_boot, 0.975), "roc_auc": roc_auc_score(yt, p),
                 "log_loss": log_loss(yt, np.clip(p, 1e-6, 1 - 1e-6)), "threshold": t,
                 "precision": precision_score(yt, p >= t, zero_division=0), "recall": recall_score(yt, p >= t),
                 "f1": f1_score(yt, p >= t)})
strict_table = pd.DataFrame(rows).set_index("model")
strict_table.round(4).to_csv(RESULTS / "strict_cutoff_metrics.csv")

a, b = probs["Strict billing + demographics + listening"], probs["Strict billing + demographics"]
diff = np.array([average_precision_score(yt[i], a[i]) - average_precision_score(yt[i], b[i]) for i in boot_idx])
listening_gain = {"mean": float(average_precision_score(yt, a) - average_precision_score(yt, b)),
                  "low": float(np.quantile(diff, 0.025)), "high": float(np.quantile(diff, 0.975))}
print(f"PR-AUC gain from listening under the strict cutoff: {listening_gain['mean']:+.4f} "
      f"(paired 95% CI {listening_gain['low']:+.4f} to {listening_gain['high']:+.4f})")
strict_table.round(4)

PR-AUC gain from listening under the strict cutoff: +0.1474 (paired 95% CI +0.1426 to +0.1524)


,pr_auc,pr_auc_low,pr_auc_high,roc_auc,log_loss,threshold,precision,recall,f1
model,,,,,,,,,
Strict billing + demographics,0.4208,0.4134,0.4276,0.8023,0.2360,0.20,0.3434,0.4299,0.3818
Strict billing + demographics + listening,0.5682,0.5610,0.5757,0.8614,0.2017,0.27,0.6279,0.4539,0.5269
Rule: auto-renew off (strict),0.1418,0.1381,0.1457,0.5553,0.2917,0.45,0.8396,0.0523,0.0984
Majority (prior),0.0899,0.0886,0.0912,0.5000,0.3024,0.45,0.0000,0.0000,0.0000
Original champion (31 March billing),0.9644,0.9624,0.9662,0.9958,0.0452,0.45,0.8828,0.9129,0.8976


## Save

In [6]:
out = {"cutoff": str(STRICT.date()), "has_tx_share": float(strict.has_tx.mean()),
       "overlap": [{"march_tx": bool(k), "n": int(r.n), "share": float(r.share), "churn_rate": float(r.churn_rate)} for k, r in overlap.iterrows()],
       "metrics": strict_table.round(4).reset_index().to_dict(orient="records"), "listening_gain": listening_gain}
(RESULTS / "strict_cutoff.json").write_text(json.dumps(out, indent=1, default=float))
print("saved results/strict_cutoff.json and results/strict_cutoff_metrics.csv")

saved results/strict_cutoff.json and results/strict_cutoff_metrics.csv
